# 6장 웹부록 | 피드백 루프 30일 시뮬레이션

「영화로 보는 AI 리터러시」 6장 '범죄를 예측하는 AI'의 코랩 실습입니다.

책의 **실습 1(종이 위의 피드백 루프)**을 컴퓨터로 1,000번 되풀이해 봅니다. 끝에는 **실습 2(기준선)**를 슬라이더로 움직여 봅니다.

**사용법**
- 위에서부터 차례로 각 칸 왼쪽의 ▶(실행) 버튼을 누르세요. 코드를 몰라도 괜찮습니다.
- `# ← 바꿔 보세요`라고 적힌 숫자만 고쳐서 다시 ▶를 누르면 됩니다.
- 처음부터 다시 하고 싶으면 메뉴에서 **런타임 → 모두 실행**을 누르세요.
- 걸리는 시간: 약 20분

## 1단계. 30일을 한 번 돌려 보기

실습 1과 같은 규칙입니다.

- 두 동네 A와 B의 실제 범죄 가능성은 **똑같이 50%**입니다(주사위 1~3).
- 첫날 A에만 기록이 1건 있습니다.
- 순찰차는 매일 **기록이 더 많은 동네**로 갑니다. 기록이 같으면 동전을 던져 정합니다.
- 순찰차가 간 동네에서만 범죄가 발견되고 기록됩니다.

아래 칸을 실행하면 30일 동안의 기록표가 나옵니다. 책의 실습 1 표와 같은 모양입니다.

In [ ]:
import random
import pandas as pd

def simulate(days=30, explore=0.0, seed=None, log=False):
    rng = random.Random(seed)
    true_rate = {"A": 0.5, "B": 0.5}   # 두 동네의 실제 범죄 가능성은 같습니다
    recorded = {"A": 1, "B": 0}        # 첫날 A에만 기록이 1건 있습니다
    rows = []
    for day in range(1, days + 1):
        if rng.random() < explore:                 # explore 비율만큼은 무작위 순찰
            target = rng.choice(["A", "B"])
        elif recorded["A"] == recorded["B"]:       # 기록이 같으면 동전 던지기
            target = rng.choice(["A", "B"])
        else:                                      # 나머지는 기록이 많은 곳으로
            target = max(recorded, key=recorded.get)
        dice = rng.randint(1, 6)
        found = dice <= 3                          # 1~3이 나오면 범죄 발견 (50%)
        if found:
            recorded[target] += 1
        rows.append({"날": day, "순찰 간 곳": target, "주사위": dice,
                     "A 기록": recorded["A"], "B 기록": recorded["B"]})
    if log:
        return pd.DataFrame(rows).set_index("날")
    return recorded

table = simulate(days=30, seed=1, log=True)   # seed를 바꾸면 다른 30일이 나옵니다
table

표를 보면 순찰차가 거의 A에만 갑니다. B에서는 범죄가 일어나도 아무도 보지 못하니 기록이 늘지 않습니다.

## 2단계. 1,000번 되풀이하기

한 번은 우연일 수 있습니다. 같은 30일을 1,000번 돌려서 평균을 냅니다. 'B의 기록이 끝까지 0건'인 경우도 세어 봅니다.

👉 **바꿔 볼 곳**: `EXPLORE`는 무작위 순찰 비율입니다. `0.0` → `0.1` → `0.3`으로 바꾸고 다시 실행해 보세요.
(책의 실습 1 '5일에 한 번은 동전'은 `EXPLORE = 0.2`와 비슷합니다.)

In [ ]:
EXPLORE = 0.0   # ← 이 숫자를 바꿔 보세요 (0.1, 0.3 등)
RUNS = 1000

runs = [simulate(explore=EXPLORE, seed=i) for i in range(RUNS)]
avg_a = sum(r["A"] for r in runs) / RUNS
avg_b = sum(r["B"] for r in runs) / RUNS
b_zero = sum(1 for r in runs if r["B"] == 0)

print(f"무작위 순찰 비율 EXPLORE = {EXPLORE}")
print(f"30일 뒤 평균 기록   A: {avg_a:.1f}건   B: {avg_b:.1f}건")
print(f"B의 기록이 0건으로 끝난 경우: {b_zero} / {RUNS}번")

## 3단계. 세 가지 EXPLORE 값을 막대그래프로 비교하기

EXPLORE가 0, 0.1, 0.3일 때 A와 B의 평균 기록을 나란히 그립니다.
두 동네의 실제 범죄는 똑같으니, 공정한 기록이라면 두 막대의 높이가 비슷해야 합니다.

(그래프 글자는 코랩에서 한글이 깨지지 않도록 영어로 적었습니다.)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

settings = [0.0, 0.1, 0.3]   # ← 비교할 값을 바꾸거나 더해 볼 수 있습니다
mean_a, mean_b = [], []
for e in settings:
    rs = [simulate(explore=e, seed=i) for i in range(1000)]
    mean_a.append(sum(r["A"] for r in rs) / len(rs))
    mean_b.append(sum(r["B"] for r in rs) / len(rs))

x = np.arange(len(settings))
plt.figure(figsize=(7, 4))
plt.bar(x - 0.2, mean_a, width=0.4, label="Town A", color="#d9534f")
plt.bar(x + 0.2, mean_b, width=0.4, label="Town B", color="#5b8def")
plt.xticks(x, [f"EXPLORE = {e}" for e in settings])
plt.ylabel("Average records after 30 days")
plt.title("Same real crime rate, different records")
plt.legend()
plt.show()

for e, a, b in zip(settings, mean_a, mean_b):
    print(f"EXPLORE {e}:  A {a:.1f}건,  B {b:.1f}건,  B의 몫 {b/(a+b):.0%}")

무작위 순찰을 조금만 섞어도 B의 모습이 데이터에 다시 들어옵니다. 다만 그만큼 '기록이 많은 곳'에 가는 날은 줄어듭니다. 이것이 무작위 순찰의 대가입니다.

## 더 해 보기. 기준선을 움직여 보기 (실습 2)

책의 실습 2에 나온 열 명(A~J)의 재범 위험 점수입니다. 1은 실제로 다시 범죄를 저지른 사람(O), 0은 아닌 사람(X)입니다.

기준선 이상인 사람에게 경보가 울립니다. 아래 칸은 기준선 0.5와 0.8의 혼동행렬, 정밀도, 재현율을 계산합니다. 책 8절 끝의 해답과 비교해 보세요.

- **정밀도** = 맞힌 경보 ÷ 전체 경보 (경보가 얼마나 믿을 만한가)
- **재현율** = 맞힌 경보 ÷ 실제로 다시 범죄를 저지른 사람 (얼마나 놓치지 않았나)

In [ ]:
people = {"A": (0.95, 1), "B": (0.90, 0), "C": (0.85, 1), "D": (0.70, 1), "E": (0.65, 0),
          "F": (0.55, 0), "G": (0.40, 1), "H": (0.30, 0), "I": (0.20, 0), "J": (0.10, 0)}

def check(threshold):
    flagged = [p for p, (s, y) in people.items() if s >= threshold]
    tp = [p for p in flagged if people[p][1] == 1]                       # 맞힌 경보
    fp = [p for p in flagged if people[p][1] == 0]                       # 헛짚은 경보
    fn = [p for p, (s, y) in people.items() if y == 1 and s < threshold]  # 놓친 사건
    actual = sum(y for s, y in people.values())
    precision = len(tp) / len(flagged) if flagged else 0
    recall = len(tp) / actual
    print(f"기준선 {threshold:.2f} → 경보 대상: {', '.join(flagged) or '없음'} ({len(flagged)}명)")
    print(f"  맞힌 경보 {len(tp)}명 {tp}   헛짚은 경보 {len(fp)}명 {fp}   놓친 사건 {len(fn)}명 {fn}")
    print(f"  정밀도 {precision:.0%}   재현율 {recall:.0%}")
    return precision, recall

check(0.5)
print()
check(0.8);

### 슬라이더로 움직이기

아래 칸을 실행하면 슬라이더가 나옵니다. 기준선을 끌어 옮기면 정밀도와 재현율이 바로 바뀝니다.
슬라이더가 보이지 않는 환경이라면, 그 아래 칸의 그래프로 같은 내용을 볼 수 있습니다.

In [ ]:
try:
    import ipywidgets as widgets
    widgets.interact(check, threshold=widgets.FloatSlider(min=0.05, max=1.0, step=0.05, value=0.5,
                                                          description="기준선"))
except ImportError:
    print("이 환경에는 슬라이더(ipywidgets)가 없습니다. 아래 그래프를 보세요.")

In [ ]:
thresholds = np.arange(0.05, 1.0001, 0.05)
prec, rec = [], []
for t in thresholds:
    flagged = [p for p, (s, y) in people.items() if s >= t]
    tp = sum(people[p][1] for p in flagged)
    prec.append(tp / len(flagged) if flagged else np.nan)
    rec.append(tp / 4)

plt.figure(figsize=(7, 4))
plt.plot(thresholds, prec, "o-", label="Precision")
plt.plot(thresholds, rec, "s-", label="Recall")
for t in (0.5, 0.8):
    plt.axvline(t, color="gray", linestyle="--")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.ylim(-0.05, 1.05)
plt.title("Moving the threshold trades precision for recall")
plt.legend()
plt.show()

## 생각해 볼 것

1. EXPLORE가 0일 때 B의 기록은 왜 거의 늘지 않을까요? 이 기록으로 AI를 학습시키면 B를 어떻게 예측할까요?
2. 무작위 순찰을 섞으면 무엇이 좋아지고, 무엇을 대가로 치릅니까? 4-2절의 '루프를 끊는 방법'과 이어서 생각해 보세요.
3. 이 점수가 '가석방 여부'를 정하는 데 쓰인다면 기준선을 어디에 두겠습니까? '상담 프로그램 안내'라면 어떻습니까? 어느 쪽 실수가 더 큰 피해를 주는지로 설명해 보세요.